# IDPFold2 — Conformational Ensemble Prediction

This notebook runs **IDPFold2**, a generative framework for modelling heterogeneous protein
thermodynamics by integrating a Mixture-of-Experts architecture into a flow-matching
framework.

**Reference:**  
[*Extending Conformational Ensemble Prediction to Multidomain Proteins and Protein Complex*](https://www.biorxiv.org/content/10.64898/2026.01.14.699584v1)

**What this notebook does:**
1. Installs all dependencies on the Colab runtime
2. Clones the IDPFold2 repository
3. Downloads the pre-trained model weights from Zenodo
4. Lets you enter a protein sequence (or use the provided example)
5. Runs inference to generate a conformational ensemble
6. Visualises the generated structures with py3Dmol

---

**Runtime requirements:** GPU (T4 or better). Go to *Runtime → Change runtime type → T4 GPU*.

## 1 · Check GPU availability

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU detected. Please go to Runtime → Change runtime type "
        "and select a GPU accelerator (T4 recommended)."
    )

gpu_name = torch.cuda.get_device_name(0)
gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU : {gpu_name}")
print(f"VRAM: {gpu_mem:.1f} GB")

## 2 · Install dependencies & clone repository

In [ ]:
import os

# Clone IDPFold2 if not already present
# TODO: Switch to https://github.com/Junjie-Zhu/IDPFold2.git (main) once the PR is merged
if not os.path.isdir("/content/IDPFold2"):
    !git clone --branch claude/setup-idpfold2-colab-pvj4N \
        https://github.com/ts387/IDPFold2.git /content/IDPFold2

os.chdir("/content/IDPFold2")
print(f"Working directory: {os.getcwd()}")

In [ ]:
# Install Python dependencies (pip-only, avoids conda on Colab)
# Note: biotite 0.41.0 requires numpy<2 (compiled against NumPy 1.x)
!pip install -q \
    "numpy<2" \
    torch-geometric==2.6.1 \
    fair-esm \
    einops==0.6 \
    dm-tree==0.1.8 \
    loguru==0.7.2 \
    hydra-core==1.3.1 \
    biotite==0.41.0 \
    biopandas==0.5.1 \
    wget==3.2 \
    cpdb-protein \
    biopython \
    rootutils \
    py3Dmol

# Install the IDPFold2 package itself
!pip install -q -e /content/IDPFold2

print("\nAll dependencies installed.")

## 3 · Download pre-trained model weights

Weights are hosted on [Zenodo (record 18239596)](https://zenodo.org/records/18239596).  
Only the EMA checkpoint is needed for inference (~400 MB).

In [ ]:
import os
import urllib.request

WEIGHTS_DIR = "/content/IDPFold2/weights"
CKPT_NAME = "IDPFold2_ema_0.999_260114.pth"
CKPT_PATH = os.path.join(WEIGHTS_DIR, CKPT_NAME)
ZENODO_URL = f"https://zenodo.org/records/18239596/files/{CKPT_NAME}"

os.makedirs(WEIGHTS_DIR, exist_ok=True)

if not os.path.isfile(CKPT_PATH):
    print(f"Downloading {CKPT_NAME} from Zenodo ...")
    !wget -q --show-progress -O "{CKPT_PATH}" "{ZENODO_URL}"
    print("Download complete.")
else:
    print(f"Checkpoint already exists at {CKPT_PATH}")

print(f"Checkpoint size: {os.path.getsize(CKPT_PATH) / 1e6:.1f} MB")

## 4 · Define input sequence

Provide a protein name and single-letter amino acid sequence below.  
The default is **α-synuclein** (140 residues), a well-studied intrinsically disordered protein.

For **multimer** inference, separate chains with `:` in both the sequence and chain IDs
fields, then set `MULTIMER = True`. Example from the authors (PDB 4MVL — transthyretin
bound to Aβ peptide):

| Field | Value |
|-------|-------|
| PROTEIN_NAME | `4mvl` |
| SEQUENCE | `QDSTSDLIPAPPLSKVPLQQNFQDNQFHGKWYVVGAAGNVLLREDKDPLKMYATIYELKEDKSYNVTSVGFDDKKCLYKIRTFVPGSQPGEFTLGRIKSEPGGTSWLVRVVSTNYNQHAMVFFKEVAQNRETFNITLYGRTKELTSELKENFIRFSKSLGLPENHIVFPVPIDQCIDGSAWSHPQFEK:DAEFRHDSGYEVHHQKLVFFAEDVGSNKGAIIGLMVGGVV` |
| CHAIN_IDS | `A:B` |
| MULTIMER | `True` |

In [ ]:
#@title Input parameters { run: "auto" }

#@markdown **Protein name** (used for output file naming)
PROTEIN_NAME = "alpha_synuclein"  #@param {type:"string"}

#@markdown **Amino acid sequence** (single-letter code; use `:` to separate chains for multimers)
SEQUENCE = "MDVFMKGLSKAKEGVVAAAEKTKQGVAEAAGKTKEGVLYVGSKTKEGVVHGVATVAEKTKEQVTNVGGAVVTGVTAVAQKTVEGAGSIAAATGFVKKDQLGKNEEGAPQEGILEDMPVDPDNEAYEMPSEEGYQDYEPEA"  #@param {type:"string"}

#@markdown **Number of conformations** to generate
NUM_SAMPLES = 50  #@param {type:"integer"}

#@markdown **Is this a multimer?** (chains separated by `:` in sequence)
MULTIMER = False  #@param {type:"boolean"}

#@markdown **Chain IDs** (only for multimers, e.g. `A:B` — must match the number of `:` separated sequences)
CHAIN_IDS = "A:B"  #@param {type:"string"}

#@markdown **Max batch length** (reduce if you hit OOM errors; increase on larger GPUs)
MAX_BATCH_LENGTH = 3000  #@param {type:"integer"}

# Strip whitespace that may creep in when pasting long sequences
SEQUENCE = "".join(SEQUENCE.split())
CHAIN_IDS = "".join(CHAIN_IDS.split())

# Validate
valid_aa = set("ACDEFGHIKLMNPQRSTVWY")
seq_clean = SEQUENCE.replace(":", "")
invalid = set(seq_clean.upper()) - valid_aa
if invalid:
    raise ValueError(f"Invalid amino acid characters found: {invalid}")

if MULTIMER:
    n_chains = len(SEQUENCE.split(":"))
    n_ids = len(CHAIN_IDS.split(":"))
    if n_chains != n_ids:
        raise ValueError(
            f"Number of chains in SEQUENCE ({n_chains}) does not match "
            f"CHAIN_IDS ({n_ids}). Separate both with ':'."
        )

total_residues = len(seq_clean)
print(f"Protein    : {PROTEIN_NAME}")
print(f"Residues   : {total_residues}")
print(f"Multimer   : {MULTIMER}")
if MULTIMER:
    print(f"Chain IDs  : {CHAIN_IDS}")
print(f"Samples    : {NUM_SAMPLES}")
print(f"Batch limit: {MAX_BATCH_LENGTH}")

In [ ]:
# Write input CSV
import pandas as pd
import os

INPUT_DIR = "/content/IDPFold2/input"
os.makedirs(INPUT_DIR, exist_ok=True)

csv_path = os.path.join(INPUT_DIR, "input_sequences.csv")

if MULTIMER:
    df = pd.DataFrame({
        "test_case": [PROTEIN_NAME],
        "chain_ids": [CHAIN_IDS],
        "sequence": [SEQUENCE],
    })
else:
    df = pd.DataFrame({
        "test_case": [PROTEIN_NAME],
        "sequence": [SEQUENCE],
    })

df.to_csv(csv_path, index=False)

print(f"Input CSV written to {csv_path}")
print(df.to_string(index=False))

## 5 · Run inference

This cell runs IDPFold2 inference. On first run it will also compute ESM2 embeddings
(cached for subsequent runs).

Approximate timings on a T4 GPU:
- ESM2 embedding extraction: ~10 s per sequence
- Structure generation: depends on sequence length and `nsamples`

In [ ]:
import os
os.chdir("/content/IDPFold2")

multimer_flag = "load_multimer=True" if MULTIMER else ""

!python src/inference.py \
    prefix={PROTEIN_NAME} \
    ckpt_dir={CKPT_PATH} \
    plm_emb_dir=/content/IDPFold2/embeddings \
    csv_dir={csv_path} \
    nsamples={NUM_SAMPLES} \
    max_batch_length={MAX_BATCH_LENGTH} \
    {multimer_flag}

In [ ]:
# Locate the output PDB
import glob

log_dirs = sorted(glob.glob(f"/content/IDPFold2/logs/{PROTEIN_NAME}_INF_*"))
if not log_dirs:
    raise FileNotFoundError("No output directory found. Check the inference log above for errors.")

latest_dir = log_dirs[-1]
pdb_files = glob.glob(os.path.join(latest_dir, "samples", "*.pdb"))
if not pdb_files:
    raise FileNotFoundError(f"No PDB files found in {latest_dir}/samples/")

OUTPUT_PDB = pdb_files[0]
print(f"Output PDB: {OUTPUT_PDB}")
print(f"File size : {os.path.getsize(OUTPUT_PDB) / 1e3:.1f} KB")

## 6 · Visualise the conformational ensemble

Uses **py3Dmol** to render the generated ensemble directly in the notebook.  
Each model in the multi-model PDB is shown as a separate conformation.

In [ ]:
import py3Dmol

with open(OUTPUT_PDB, "r") as f:
    pdb_str = f.read()

# Count models
n_models = pdb_str.count("MODEL")
print(f"Visualising {n_models} conformations")

view = py3Dmol.view(width=800, height=600)
view.addModelsAsFrames(pdb_str, "pdb")

# Style each model
for i in range(n_models):
    view.setStyle({"model": i}, {"cartoon": {"color": "spectrum", "opacity": 0.6}})

view.zoomTo()
view.show()

In [ ]:
# Animated view: cycle through conformations
view_anim = py3Dmol.view(width=800, height=600)
view_anim.addModelsAsFrames(pdb_str, "pdb")
view_anim.setStyle({}, {"cartoon": {"color": "spectrum"}})
view_anim.zoomTo()
view_anim.animate({"loop": "forward", "interval": 200})
view_anim.show()

## 7 · Quick analysis (Rg and end-to-end distance)

In [ ]:
# quick_analysis.py expects a directory containing PDB files
import os
SAMPLES_DIR = os.path.dirname(OUTPUT_PDB)
!python /content/IDPFold2/scripts/quick_analysis.py {SAMPLES_DIR}

# Load and display metrics
import pickle
metrics_path = os.path.join(SAMPLES_DIR, "metrics.pkl")
if os.path.isfile(metrics_path):
    with open(metrics_path, "rb") as f:
        metrics = pickle.load(f)
    import numpy as np
    for i, name in enumerate(metrics["name"]):
        rg_vals = metrics["rg_predict"][i]
        re2e_vals = metrics["re2e_predict"][i]
        print(f"\n--- {name} ---")
        print(f"  Rg   : mean={np.mean(rg_vals):.2f} Å, std={np.std(rg_vals):.2f} Å")
        print(f"  Re2e : mean={np.mean(re2e_vals):.2f} Å, std={np.std(re2e_vals):.2f} Å")

## 8 · (Optional) All-atom back-mapping with cg2all

Convert the Cα-only ensemble to all-atom structures using
[cg2all](https://github.com/huhlim/cg2all). This enables full molecular
visualization (cartoons, ribbons, sidechains) in ChimeraX, PyMOL, etc.

**Note:** This step takes a few minutes depending on ensemble size and sequence length.

In [ ]:
#@title Run cg2all back-mapping { run: "auto", display-mode: "form" }

#@markdown Enable to convert Cα ensemble → all-atom PDB
ENABLE_CG2ALL = True  #@param {type:"boolean"}

OUTPUT_AA_PDB = None

if ENABLE_CG2ALL:
    # Install cg2all and mdtraj (not needed for core inference)
    !pip install -q mdtraj "git+https://github.com/huhlim/cg2all"

    import os
    import mdtraj as md

    cg2all_dir = os.path.join(os.path.dirname(OUTPUT_PDB), "cg2all_tmp")
    os.makedirs(cg2all_dir, exist_ok=True)

    # Convert multi-model PDB → DCD trajectory + single-frame topology
    traj = md.load(OUTPUT_PDB)
    traj.save_dcd(os.path.join(cg2all_dir, "traj.dcd"))
    traj[0].save_pdb(os.path.join(cg2all_dir, "topology.pdb"))

    OUTPUT_AA_PDB = OUTPUT_PDB.replace(".pdb", "_allatom.pdb")

    # Run cg2all (--batch 20 / --proc 2 tuned for Colab's 2 CPU cores)
    !convert_cg2all \
        -p {cg2all_dir}/topology.pdb \
        -d {cg2all_dir}/traj.dcd \
        -o {cg2all_dir}/aa_traj.dcd \
        -opdb {OUTPUT_AA_PDB} \
        --cg CalphaBasedModel \
        --batch 20 --proc 2

    print(f"\nAll-atom PDB: {OUTPUT_AA_PDB}")
    print(f"File size   : {os.path.getsize(OUTPUT_AA_PDB) / 1e3:.1f} KB")
else:
    print("cg2all back-mapping skipped. Set ENABLE_CG2ALL = True to run.")

## 9 · Download output

Download the generated PDB file(s) to your local machine.

In [ ]:
import os
try:
    from google.colab import files
    files.download(OUTPUT_PDB)
    if OUTPUT_AA_PDB is not None and os.path.isfile(OUTPUT_AA_PDB):
        files.download(OUTPUT_AA_PDB)
except ImportError:
    print(f"Not running in Colab. Output PDB is at: {OUTPUT_PDB}")
    if OUTPUT_AA_PDB is not None and os.path.isfile(OUTPUT_AA_PDB):
        print(f"All-atom PDB is at: {OUTPUT_AA_PDB}")

---

## Notes

- **GPU memory:** The default `max_batch_length=3000` works for T4 (15 GB). If you
  encounter OOM errors, reduce this value. On A100 or V100, you can increase it to
  `6000` or higher.
- **Output:** The generated structures contain only Cα atoms. Use the optional
  **cg2all back-mapping** cell (section 8) to convert to all-atom structures for
  full visualization in ChimeraX, PyMOL, etc.
- **Multimer inference:** Set `MULTIMER = True` and separate chain sequences with `:`
  in the sequence field.
- **Reproducibility:** Random seeds are not fixed by default to produce diverse
  ensembles. Set `seed=42 deterministic=True` in the inference command for
  reproducible results.

### Citation

```bibtex
@article{zhu2026idpfold2,
  title={Extending Conformational Ensemble Prediction to Multidomain Proteins and Protein Complex},
  author={Zhu, Junjie and others},
  journal={bioRxiv},
  year={2026}
}
```